In [ ]:
# ollama pull qwen2.5:3b

import sqlite3
import json
import re
from pathlib import Path

import pandas as pd
import ollama
import geonamescache
import unicodedata

# import DKB parser temporarily to get real data
import sys
sys.path.append("../../")
from backend.parsers import DKB

gc = geonamescache.GeonamesCache()

city_names = {
    "".join(
        c for c in unicodedata.normalize("NFKD", city["name"].lower())
        if not unicodedata.combining(c)
    )
    for city in gc.get_cities().values()
}

def clean_payee(text: str) -> str:
    """
    Normalizes a payee string by:
      - converting to lowercase
      - removing accents/umlauts
      - removing URLs
      - removing standalone numbers
      - removing special characters and punctuation
      - removing city names
      - collapsing multiple spaces
    """
    if pd.isna(text):
        return ""

    # Lowercase and strip whitespace
    text = str(text).lower().strip()

    # Remove accents (ä -> a, é -> e, ...)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))

    # Remove URLs/domains
    text = re.sub(r"\.com\b|\.co\.uk\b|www\.", " ", text)

    # Remove standalone numbers
    text = re.sub(r"\b\d+\b", " ", text)

    # Remove special characters
    text = re.sub(r"[*#]", " ", text)

    # Remove punctuation (keep letters, digits and spaces)
    text = re.sub(r"[^\w\s]", " ", text)

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Remove city names
    words = text.split()
    text = " ".join(word for word in words if word not in city_names)

    return text


# ----------------------------
# SQLite memory
# ----------------------------
DB_PATH = Path("expense_memory.sqlite")

CATEGORIES = [
    "Lebensmittel",
    "Transport",
    "Miete",
    "Nebenkosten",
    "Entertainment",
    "Shopping",
    "Gesundheit",
    "Einkommen",
    "Sonstiges",
    "Hobby"
]

def init_database():
    conn = sqlite3.connect(DB_PATH)

    conn.execute("""
        CREATE TABLE IF NOT EXISTS merchant_categories (
            merchant TEXT PRIMARY KEY,
            category TEXT NOT NULL,
            confidence REAL DEFAULT 1.0
        )
    """)

    conn.commit()
    return conn


def lookup_memory(conn, description):
    """
    Look for previously classified merchants.
    """

    text = description.upper()

    rows = conn.execute(
        "SELECT merchant, category, confidence FROM merchant_categories"
    ).fetchall()

    for merchant, category, confidence in rows:
        if merchant in text:
            return {
                "category": category,
                "confidence": confidence,
                "source": "memory"
            }

    return None


def save_memory(conn, merchant, category, confidence):
    conn.execute(
        """
        INSERT OR REPLACE INTO merchant_categories
        (merchant, category, confidence)
        VALUES (?, ?, ?)
        """,
        (merchant.upper(), category, confidence)
    )

    conn.commit()

def correct_category(conn, merchant, new_category):
    """
    Manually corrects a merchant category.
    Future transactions with this merchant will use the corrected category.
    """

    if new_category not in CATEGORIES:
        raise ValueError(
            f"Unknown category: {new_category}"
        )

    conn.execute(
        """
        INSERT OR REPLACE INTO merchant_categories
        (merchant, category, confidence)
        VALUES (?, ?, ?)
        """,
        (
            merchant.upper(),
            new_category,
            1.0
        )
    )

    conn.commit()


# ----------------------------
# Rule engine
# ----------------------------

RULES = {
    "Lebensmittel": [
        "rewe",
        "lidl",
        "aldi",
        "edeka",
        "netto",
        "penny",
        "markant"
    ],
    "Transport": [
        "db ",
        "deutsche bahn",
        "flix",
        "uber",
        "shell",
        "aral",
    ],
    "Entertainment": [
        "netflix",
        "spotify",
        "steam",
        "disney",
    ],
    "Nebenkosten": [
        "telekom",
        "vodafone",
        "ENERGY",
        "STROM",
    ],
    "Gesundheit": [
        "apotheke",
        "pharmacy",
        "arzt",
    ],
    "Hobby": [
        "wolle"
    ]
}


def apply_rules(description):

    if pd.isna(description):
        return None

    text = description.lower()

    for category, keywords in RULES.items():

        for keyword in keywords:

            if keyword.lower() in text:
                return {
                    "category": category,
                    "confidence": 0.95,
                    "source": "rules"
                }

    return None


# ----------------------------
# LLM fallback
# ----------------------------

def classify_with_llm(description):

    prompt = f"""
You are a personal finance assistant.

Your task is to categorize ONE bank transaction.

Categories:
- Lebensmittel: supermarkets, groceries, food stores
- Transport: trains, buses, fuel, taxis
- Miete: rent payments
- Nebenkosten: electricity, internet, phone, utilities
- Entertainment: streaming, games, movies
- Shopping: clothes, online shopping, general purchases
- Gesundheit: pharmacies, doctors, medical expenses
- Einkommen: salary, refunds, incoming payments
- Hobby: hobbies, crafts, sports equipment
- Sonstige: only if nothing else fits

Analyze the merchant name carefully.

Transaction:
{description}

Examples:

"REWE SAGT DANKE"
=> Lebensmittel

"DEUTSCHE BAHN AG"
=> Transport

"NETFLIX.COM"
=> Entertainment

"TK MAXX"
=> Shopping

Return ONLY JSON:

{{
    "category": "one category from the list",
    "confidence": 0.0
}}
"""

    response = ollama.chat(
        model="qwen2.5:3b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        format="json"   # important
    )

    result = json.loads(
        response["message"]["content"]
    )

    return {
        "category": result["category"],
        "confidence": float(result["confidence"]),
        "source": "llm"
    }


# ----------------------------
# Merchant extraction
# ----------------------------

def extract_merchant(description):
    """
    Crude normalization.
    Improve this over time.
    """

    text = description.upper()

    text = re.sub(
        r"\d+",
        "",
        text
    )

    text = re.sub(
        r"[^A-ZÄÖÜ ]",
        " ",
        text
    )

    words = text.split()

    return " ".join(words[:2])


# ----------------------------
# Main classifier
# ----------------------------

def classify_expenses(df):

    conn = init_database()

    results = []

    for _, row in df.iterrows():

        description = row["payee"]


        result = (
            lookup_memory(conn, description)
            or
            apply_rules(description)
            or
            classify_with_llm(clean_payee(description))
        )


        merchant = clean_payee(description)


        # Remember LLM/rule decisions
        save_memory(
            conn,
            merchant,
            result["category"],
            result["confidence"]
        )


        results.append(result)


    output = df.copy()

    output["category"] = [
        r["category"]
        for r in results
    ]

    output["confidence"] = [
        r["confidence"]
        for r in results
    ]

    output["classification_source"] = [
        r["source"]
        for r in results
    ]

    conn.close()

    return output

def correct_transaction(df, index, new_category):

    conn = init_database()

    description = df.loc[index, "payee"]

    merchant = clean_payee(description)

    correct_category(
        conn,
        merchant,
        new_category
    )

    conn.close()

In [ ]:
import pandas as pd
# from expense_classifier import classify_expenses

transactions = DKB.load("../../data/07-07-2026_Umsatzliste_Girokonto_DE60120300001200204400 (3).csv")
transactions["payee"] = transactions["payee"].apply(clean_payee)
transactions = transactions[["payee"]].head(20)

transactions = pd.read_csv("../../data/test_data.csv")
transactions = transactions[["payee"]].head(20)

result = classify_expenses(transactions)

print(result)

In [ ]:
correct_transaction(
    transactions,
    0,
    "Abrechnung"
)

correct_transaction(
    transactions,
    15,
    "Kleidung"
)